# 03 Service Metric Profiles


This notebook builds the profile-based service-metric branch for `CCT` and abandonment.

These metrics are handled differently from `CV` because they are not clean additive counts. In the audit, daily and interval `CCT`/abandonment did not reconcile exactly even on complete interval days, so the interval sheets are used to learn intraday shape while the daily sheets remain the authoritative level.

## Core Implementation Used Here

The cells below call `src/pipeline.py` so the notebooks and command-line runner stay consistent. For reviewability, this section shows the exact source code for the functions used in this notebook.

```python
def build_cct_profiles(daily_all: pd.DataFrame, interval_all: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    scale_rows = []
    train_daily = daily_all[(daily_all["Date"] >= TRAIN_START) & (daily_all["Date"] <= TRAIN_END)]
    train_interval = interval_all[(interval_all["Date"] >= TRAIN_START) & (interval_all["Date"] <= TRAIN_END)].copy()
    train_interval["dow"] = train_interval["Date"].dt.dayofweek

    for portfolio in PORTFOLIOS:
        p_int = train_interval[
            (train_interval["Portfolio"] == portfolio)
            & train_interval["CCT"].notna()
            & train_interval["Call Volume"].fillna(0).ge(3)
        ].copy()
        p_daily = train_daily[train_daily["Portfolio"] == portfolio]
        avg_daily_cct = float(p_daily["CCT"].dropna().mean())
        scale_rows.append({"Portfolio": portfolio, "avg_train_daily_cct": avg_daily_cct})
        fallback = p_int.groupby("slot")["CCT"].median().reindex(range(SLOTS_PER_DAY)).interpolate().bfill().ffill()
        fallback_arr = gaussian_filter1d(fallback.to_numpy(dtype=float), sigma=0.7)
        for dow in range(7):
            sub = p_int[p_int["dow"] == dow]
            arr = sub.groupby("slot")["CCT"].median().reindex(range(SLOTS_PER_DAY)).to_numpy(dtype=float)
            arr = np.where(np.isfinite(arr), arr, fallback_arr)
            arr = gaussian_filter1d(arr, sigma=0.7)
            for slot, value in enumerate(arr):
                rows.append({"Portfolio": portfolio, "dow": dow, "slot": slot, "cct_profile": float(value)})
    return pd.DataFrame(rows), pd.DataFrame(scale_rows)

def build_abandonment_profiles(interval_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    train_interval = interval_all[(interval_all["Date"] >= TRAIN_START) & (interval_all["Date"] <= TRAIN_END)].copy()
    train_interval = train_interval[train_interval["Abandoned Calls"].notna()].copy()
    train_interval["dow"] = train_interval["Date"].dt.dayofweek
    daily_ab = train_interval.groupby(["Portfolio", "Date"])["Abandoned Calls"].sum().rename("daily_observed_abandoned_calls")
    train_interval = train_interval.merge(daily_ab.reset_index(), on=["Portfolio", "Date"], how="left")
    train_interval = train_interval[train_interval["daily_observed_abandoned_calls"] > 0].copy()
    train_interval["abd_share"] = train_interval["Abandoned Calls"] / train_interval["daily_observed_abandoned_calls"]

    for portfolio in PORTFOLIOS:
        p_int = train_interval[train_interval["Portfolio"] == portfolio]
        fallback = smooth_share(p_int.groupby("slot")["abd_share"].median().reindex(range(SLOTS_PER_DAY), fill_value=0).to_numpy())
        for dow in range(7):
            sub = p_int[p_int["dow"] == dow]
            arr = sub.groupby("slot")["abd_share"].median().reindex(range(SLOTS_PER_DAY)).to_numpy(dtype=float)
            arr = np.where(np.isfinite(arr), arr, fallback)
            arr = smooth_share(arr)
            for slot, value in enumerate(arr):
                rows.append({"Portfolio": portfolio, "dow": dow, "slot": slot, "abd_share_profile": float(value)})
    return pd.DataFrame(rows)

def forecast_service_metrics(anchors: pd.DataFrame, cct_profiles: pd.DataFrame, cct_scale: pd.DataFrame, abd_profiles: pd.DataFrame) -> pd.DataFrame:
    cct_lookup = cct_profiles.set_index(["Portfolio", "dow", "slot"])["cct_profile"]
    abd_lookup = abd_profiles.set_index(["Portfolio", "dow", "slot"])["abd_share_profile"]
    scale_lookup = cct_scale.set_index("Portfolio")["avg_train_daily_cct"]
    rows = []
    for _, anchor in anchors.sort_values(["Portfolio", "Date"]).iterrows():
        portfolio = anchor["Portfolio"]
        dt = anchor["Date"]
        dow = int(dt.dayofweek)
        daily_cct = float(anchor["CCT"])
        avg_train_cct = float(scale_lookup.loc[portfolio])
        daily_abd = float(anchor["daily_abandoned_calls"])
        cct_scale_factor = daily_cct / avg_train_cct if avg_train_cct > 0 else 1.0
        abd_share = normalize([abd_lookup.loc[(portfolio, dow, slot)] for slot in range(SLOTS_PER_DAY)])
        for slot in range(SLOTS_PER_DAY):
            rows.append(
                {
                    "Portfolio": portfolio,
                    "Date": dt,
                    "slot": slot,
                    "interval_cct": float(cct_lookup.loc[(portfolio, dow, slot)] * cct_scale_factor),
                    "interval_abandoned_calls": float(daily_abd * abd_share[slot]),
                    "abd_share": float(abd_share[slot]),
                }
            )
    return pd.DataFrame(rows)

def run_service_pipeline() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ensure_dirs()
    daily_all = pd.read_csv(PROCESSED_DIR / "daily_clean.csv", parse_dates=["Date"])
    interval_all = pd.read_csv(PROCESSED_DIR / "interval_clean.csv", parse_dates=["Date"])
    anchors = pd.read_csv(PROCESSED_DIR / "august_daily_anchors.csv", parse_dates=["Date"])
    cct_profiles, cct_scale = build_cct_profiles(daily_all, interval_all)
    abd_profiles = build_abandonment_profiles(interval_all)
    service_forecast = forecast_service_metrics(anchors, cct_profiles, cct_scale, abd_profiles)
    cct_profiles.to_csv(PROCESSED_DIR / "cct_profiles.csv", index=False)
    cct_scale.to_csv(PROCESSED_DIR / "cct_profile_scaling.csv", index=False)
    abd_profiles.to_csv(PROCESSED_DIR / "abandonment_share_profiles.csv", index=False)
    service_forecast.to_csv(OUTPUT_DIR / "service_interval_forecast_unbiased.csv", index=False)
    return cct_profiles, cct_scale, abd_profiles, service_forecast
```


In [ ]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'pipeline.py').exists():
        ROOT = candidate
        break
    if (candidate / 'datathon_final' / 'src' / 'pipeline.py').exists():
        ROOT = candidate / 'datathon_final'
        break
if ROOT is None:
    raise RuntimeError('Could not find datathon_final project root.')
sys.path.insert(0, str(ROOT))
from src import pipeline


## Build CCT and abandonment profiles

For `CCT`, the historical profile is the median `CCT` by portfolio, day of week, and slot, smoothed across adjacent intervals. For abandonment, each historical day is converted into slot-level shares of that day's observed abandoned calls; those shares are then summarized by portfolio, day of week, and slot.

In [ ]:
cct_profiles, cct_scale, abd_profiles, service_forecast = pipeline.run_service_pipeline()
print(f'CCT profile rows: {len(cct_profiles):,}')
print(f'Abandonment profile rows: {len(abd_profiles):,}')
print(f'Service forecast rows: {len(service_forecast):,}')

## Profile scaling logic

`CCT` is not a share. For August, the intraday CCT curve is scaled up or down to match the daily CCT anchor. Abandonment is share-based: August daily abandoned calls are computed from daily `CV × Abandon Rate`, allocated across slots by the abandonment share profile, and then converted back into interval abandon rate after combining with interval `CV`.

In [ ]:
display(cct_scale)
display(cct_profiles.head())
display(abd_profiles.head())